# 02 · Preprocessing and split verification

Checks the manifest produced by `python -m SRC.data.prepare_dataset`, verifies the stratified
70/15/15 split and confirms that no duplicate group straddles two splits. Also shows what the
training augmentation does to a sample image.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
manifest = pd.read_csv(PROJECT_ROOT / 'Dataset/dataset_manifest.csv')
print(manifest.shape)
manifest.head(3)

In [ ]:
table = pd.crosstab(manifest['class_name'], manifest['split'])[['train', 'val', 'test']]
print('totals:', manifest['split'].value_counts().to_dict())
print('duplicate groups spanning splits:', int(manifest.groupby('dup_group')['split'].nunique().gt(1).sum()))
table

## Model input preprocessing
`preprocess_for_model` is the single source of truth: PIL bilinear resize to 224×224 in the
0–255 range. MobileNetV2's own scaling lives inside the saved model.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from PIL import Image, ImageOps
from SRC.utils.imaging import preprocess_for_model

row = manifest[manifest['split'] == 'train'].iloc[0]
with Image.open(PROJECT_ROOT / row['relative_path']) as img:
    rgb = np.asarray(ImageOps.exif_transpose(img).convert('RGB'))

processed = preprocess_for_model(rgb, 224)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(rgb); axes[0].set_title(f'original {rgb.shape[:2]}')
axes[1].imshow(processed.astype('uint8')); axes[1].set_title('224x224 model input')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

## Training-only augmentation
Validation and test arrays are cached without augmentation; only the training pipeline augments.

In [ ]:
import tensorflow as tf
from SRC.models.train import build_augmentation

augment = build_augmentation()
batch = tf.constant(processed[None, ...] / 255.0)
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax in axes:
    view = augment(batch, training=True)[0].numpy()
    ax.imshow(np.clip(view, 0, 1)); ax.axis('off')
axes[0].set_title('augmented samples')
plt.tight_layout()